In [1]:
from sympy.tensor.tensor import (
    TensorIndexType, 
    TensorHead, 
    TensorIndex,
    tensor_indices,
)
from sympy import symbols, Derivative, diff, Expr, expand, Add, solve
from typing import List, Tuple

In [2]:
index_type = TensorIndexType("Coords")
i, j, k = tensor_indices('i, j, k', index_type)

t = symbols('t', real=True)
y = TensorHead('y', [index_type])
gamma = TensorHead(r'\Gamma', [index_type, index_type, index_type])

display(index_type)
display(gamma)

Coords

\Gamma(Coords,Coords,Coords)

In [3]:
def _walk_find_indexed_tensorheads(expr: Expr) -> List[Tuple[Expr, TensorHead, Tuple[TensorIndex, ...]]]:
    results = []

    if hasattr(expr, 'components'):        
        if len(expr.components) == 1:
            head = expr.components[0]
            indices = expr.args[1]
            results.append((expr, head, indices))
    
    for arg in expr.args:
        results.extend(_walk_find_indexed_tensorheads(arg))
    
    return results

def _recursive_apply_deriv_to_geod_taylor(exprs: List[Expr], idx_counter: int, deriv_order: int, gamma_deriv_orders: List[TensorHead]):
    # print(f"at level {deriv_order}...")

    if deriv_order == 0:
        return exprs

    all_terms = []

    for expr in exprs:
        expr_deriv_chain_terms = []
        indexed_tensors = _walk_find_indexed_tensorheads(expr)

        # print("----------------------")
        # print(f"term: {expr}")
        for tensor_expr, head, indices in indexed_tensors:
            # print(f"tensor expr: {tensor_expr}")

            # differentiating one of the terms
            new_expr = expr.copy()

            # print(f"tensor_expr: {tensor_expr}")

            if head == y:
                k, = indices
                i, j = tensor_indices(f'i{idx_counter}:{idx_counter+2}', index_type)
                idx_counter += 2

                y_accel = -gamma(k, -i, -j) * y(i) * y(j)
                new_expr = new_expr.replace(tensor_expr, y_accel)
                
            elif head in gamma_deriv_orders:
                head_idx_in_deriv_orders = gamma_deriv_orders.index(head)
                head_deriv_idx_in_deriv_orders = head_idx_in_deriv_orders + 1

                gamma_deriv = None
                if head_deriv_idx_in_deriv_orders < len(gamma_deriv_orders):
                    # derivative already exists
                    gamma_deriv = gamma_deriv_orders[head_deriv_idx_in_deriv_orders]
                else:
                    # create the derivative
                    gamma_deriv = TensorHead(rf"\Gamma_{head_deriv_idx_in_deriv_orders}", [index_type for _ in range(3 + head_deriv_idx_in_deriv_orders)])
                    gamma_deriv_orders.append(gamma_deriv)
                

                l = tensor_indices(f'{idx_counter}', index_type)
                idx_counter += 1

                chain_gamma_deriv = gamma_deriv(*indices, -l) * y(l)
                new_expr = new_expr.replace(tensor_expr, chain_gamma_deriv)

            # print(f"new term: {new_expr}")

            expr_deriv_chain_terms.append(new_expr)
        # print("---------------------------")

        all_terms.extend(expr_deriv_chain_terms)

    return _recursive_apply_deriv_to_geod_taylor(all_terms, idx_counter, deriv_order - 1, gamma_deriv_orders)

def apply_deriv_to_geod_taylor(expr, deriv_order: int):
    terms = _recursive_apply_deriv_to_geod_taylor([expr], 0, deriv_order, [gamma])
    return sum(terms)


In [14]:
result = apply_deriv_to_geod_taylor(y(k), 2)
# print("final result")
display(result)

-\Gamma_1(k, -C_1, -C_2, -C_0)*y(C_0)*y(C_1)*y(C_2) + \Gamma(k, -C_0, -C_1)*y(C_0)*\Gamma(C_1, -C_2, -C_3)*y(C_2)*y(C_3) + \Gamma(k, -C_0, -C_3)*\Gamma(C_0, -C_1, -C_2)*y(C_1)*y(C_2)*y(C_3)

In [5]:
solve(result, y)

/home/samuels/Documents/ece602-mfld-optim-taylor-approx/.venv/lib/python3.14/site-packages/sympy/utilities/iterables.py:3075: SymPyDeprecationWarning: 

The data attribute of TensorIndexType is deprecated. Use The
replace_with_arrays() method instead.

See https://docs.sympy.org/latest/explanation/active-deprecations.html#deprecated-tensorindextype-attrs
for details.

This has been deprecated since SymPy version 1.4. It
will be removed in a future version of SymPy.

  iter(i)
/home/samuels/Documents/ece602-mfld-optim-taylor-approx/.venv/lib/python3.14/site-packages/sympy/tensor/tensor.py:2739: SymPyDeprecationWarning: 

The data attribute of TensorIndexType is deprecated. Use The
replace_with_arrays() method instead.

See https://docs.sympy.org/latest/explanation/active-deprecations.html#deprecated-tensorindextype-attrs
for details.

This has been deprecated since SymPy version 1.4. It
will be removed in a future version of SymPy.

  if not self.data:


ValueError: No iteration on abstract tensors